In [1]:
!pip install -q transformers datasets evaluate seqeval matplotlib seaborn scikit-learn pytorch-crf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00


# Load library

In [2]:
# Load library
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from torchcrf import CRF
from IPython.display import display

# Config


In [3]:
# config
CHOSEN_MODEL = 'roberta-base'
DATASET_FILE = '/content/dataset.jsonl'

RUN_MODE = 'ALL'

#Load and Prepare data

In [4]:
# load and prepare data
dataset = load_dataset('json', data_files=DATASET_FILE, split='train')
dataset = dataset.train_test_split(test_size=0.2, seed=42)

all_tags = [tag for tags_list in dataset["train"]["ner_tags"] for tag in tags_list]
unique_tags = sorted(list(set(all_tags)))

label2id = {label: i for i, label in enumerate(unique_tags)}
id2label = {i: label for i, label in enumerate(unique_tags)}

def encode_tags(example):
  example["ner_tags_id"] = [label2id.get(tag, 0) for tag in example["ner_tags"]]
  return example

dataset = dataset.map(encode_tags)

# cal class weight for focal loss
tag_counts = pd.Series(all_tags).value_counts()
total_tags = len(all_tags)
class_weights = [total_tags / (len(unique_tags) * tag_counts[label]) for label in unique_tags]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/8800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2201 [00:00<?, ? examples/s]

# Define losses and train model

In [5]:
# define losses and trainers
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss_raw = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss_raw)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss_raw
        if self.weight is not None:
            alpha = self.weight[targets]
            focal_loss = focal_loss * alpha
        return focal_loss.mean()

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        active_loss = inputs["attention_mask"].view(-1) == 1
        active_logits = logits.view(-1, model.config.num_labels)
        active_labels = torch.where(active_loss, labels.view(-1), torch.tensor(-100, device=labels.device))

        valid_indices = active_labels != -100
        valid_logits = active_logits[valid_indices]
        valid_labels = active_labels[valid_indices]

        focal_loss_fn = FocalLoss(weight=class_weights_tensor.to(model.device), gamma=2.0)
        loss = focal_loss_fn(valid_logits, valid_labels)
        return (loss, outputs) if return_outputs else loss

# DICE LOSS
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6, ignore_index=-100):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        num_classes = logits.shape[-1]
        probs = F.softmax(logits, dim=-1).view(-1, num_classes)
        targets = targets.view(-1)

        valid_mask = targets != self.ignore_index
        valid_targets = targets[valid_mask]
        valid_probs = probs[valid_mask]

        if valid_targets.numel() == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        targets_one_hot = F.one_hot(valid_targets, num_classes=num_classes).float()
        intersection = torch.sum(valid_probs * targets_one_hot, dim=0)
        cardinality = torch.sum(valid_probs + targets_one_hot, dim=0)

        dice_score = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return (1. - dice_score).mean()

class DiceLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        dice_loss_fn = DiceLoss(ignore_index=-100)
        loss = dice_loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

#  CE + DICE LOSS TRAINER
class CEDiceLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

        labels = inputs.get("labels")

        outputs = model(**inputs)

        ce_loss = outputs.loss
        logits = outputs.logits

        dice_loss_fn = DiceLoss(ignore_index=-100)
        dice_loss = dice_loss_fn(logits, labels)

        total_loss = ce_loss + dice_loss

        return (total_loss, outputs) if return_outputs else total_loss

# CRF Architechture

In [6]:
# CRF architechture
class TransformerCRF(nn.Module):
    def __init__(self, model_name, num_labels, use_focal_loss=False, gamma=2.0):
        super(TransformerCRF, self).__init__()
        self.num_labels = num_labels
        self.use_focal_loss = use_focal_loss
        self.gamma = gamma
        self.config = AutoConfig.from_pretrained(model_name)
        self.transformer = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.classifier(outputs.last_hidden_state)

        loss = None
        if labels is not None:
            mask = attention_mask.bool()
            safe_labels = torch.where(labels == -100, torch.tensor(0, device=labels.device), labels)

            if self.use_focal_loss:
                # Sequence-level Focal Loss
                nll = -self.crf(logits, tags=safe_labels, mask=mask, reduction='none')
                pt = torch.exp(-nll)
                loss = (((1 - pt) ** self.gamma) * nll).mean()
            else:
                # Standard CRF NLL Loss
                loss = -self.crf(logits, tags=safe_labels, mask=mask, reduction='mean')

        eval_mask = attention_mask.bool()
        decoded_tags = self.crf.decode(logits, mask=eval_mask)

        fake_logits = torch.zeros_like(logits)
        for i, tags in enumerate(decoded_tags):
            for j, tag in enumerate(tags):
                fake_logits[i, j, tag] = 1.0

        return {"loss": loss, "logits": fake_logits} if loss is not None else {"logits": fake_logits}

# Evaluation metrics

In [7]:
# evaluation metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Training engine

In [8]:
# core trainine engine
def run_experiment(model_name, arch_type):
    base_name = model_name.split('-')[0].upper()
    arch_name = f"{base_name} + {arch_type}"
    print(f"\nrunning {arch_name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True if "roberta" in model_name else False)

    def tokenize_and_align_labels(examples):
        tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        labels = []
        for i, label in enumerate(examples["ner_tags_id"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            previous_word_idx = None
            label_ids = []
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(label[word_idx])
                else:
                    label_ids.append(-100)
                previous_word_idx = word_idx
            labels.append(label_ids)
        tokenized_inputs["labels"] = labels
        return tokenized_inputs

    tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    if arch_type == "Base_CE":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(unique_tags), id2label=id2label, label2id=label2id
        )
        trainer_class = Trainer
    elif arch_type == "FocalLoss":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(unique_tags), id2label=id2label, label2id=label2id
        )
        trainer_class = FocalLossTrainer
    elif arch_type == "DiceLoss":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(unique_tags), id2label=id2label, label2id=label2id
        )
        trainer_class = DiceLossTrainer
    elif arch_type == "CE+DiceLoss":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(unique_tags), id2label=id2label, label2id=label2id
        )
        trainer_class = CEDiceLossTrainer
    elif arch_type == "CRF":
        model = TransformerCRF(model_name, num_labels=len(unique_tags), use_focal_loss=False)
        trainer_class = Trainer
    elif arch_type == "CRF+FocalLoss":
        model = TransformerCRF(model_name, num_labels=len(unique_tags), use_focal_loss=True, gamma=2.0)
        trainer_class = Trainer

    training_args = TrainingArguments(
        output_dir=f"./temp_{arch_name.replace(' ', '_').replace('+', '_')}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none"
    )

    trainer = trainer_class(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_results = trainer.evaluate()
    f1_score = eval_results['eval_f1']
    print(f"Done {arch_name}! F1: {f1_score:.4f}")

    return {
        "Architecture": arch_type,
        "F1-Score": f1_score,
        "Precision": eval_results['eval_precision'],
        "Recall": eval_results['eval_recall'],
        "Accuracy": eval_results['eval_accuracy']
    }

# Execution

In [9]:
#execution and display

print(f"Model type: {CHOSEN_MODEL.upper()}")
print(f"Run mode: {RUN_MODE}")

VALID_ARCHS = ["Base_CE", "FocalLoss", "DiceLoss", "CE+DiceLoss", "CRF", "CRF+FocalLoss"]

if RUN_MODE == "ALL":
    architectures = VALID_ARCHS
elif RUN_MODE in VALID_ARCHS:
    architectures = [RUN_MODE]
else:
    raise ValueError(f"run mode inn't valid, choose all or one in: {VALID_ARCHS}")

results_list = []

for arch in architectures:
    try:
        res = run_experiment(CHOSEN_MODEL, arch_type=arch)
        results_list.append(res)
    except Exception as e:
        print(f"Error {CHOSEN_MODEL} + {arch}: {e}")

# Hiển thị bảng kết quả
print("\n" + "*"*50)
print(f"Evaluate for {CHOSEN_MODEL.upper()}")
print("*"*50)

df_results = pd.DataFrame(results_list)

if not df_results.empty:
    df_results = df_results.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
    display(df_results)

    best_arch = df_results.iloc[0]['Architecture']
    best_f1 = df_results.iloc[0]['F1-Score']

    if len(architectures) > 1:
        print(f"\nBest architechture {best_arch} (F1: {best_f1:.4f}) ")
    else:
        print(f"\nDone test architechture: {best_arch} (F1: {best_f1:.4f})")

Model type: DISTILBERT-BASE-CASED
Run mode: ALL

running DISTILBERT + Base_CE...


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,6.063321,1.893336,0.872383,0.899621,0.885792,0.967935
2,1.626791,1.699709,0.889768,0.902101,0.895892,0.970746
3,1.165741,1.622400,0.889589,0.906332,0.897882,0.971655
4,1.004892,1.594515,0.894037,0.912168,0.903011,0.973283
5,0.757974,1.615234,0.898232,0.911730,0.904931,0.973600


Done DISTILBERT + CRF+FocalLoss! F1: 0.9049

**************************************************
Evaluate for DISTILBERT-BASE-CASED
**************************************************


,Architecture,F1-Score,Precision,Recall,Accuracy
0,CE+DiceLoss,0.910913,0.906634,0.915232,0.974551
1,Base_CE,0.909526,0.904604,0.914502,0.974382
2,CRF,0.906390,0.900259,0.912606,0.973790
3,CRF+FocalLoss,0.904931,0.898232,0.911730,0.973600
4,DiceLoss,0.891051,0.893733,0.888386,0.968548
5,FocalLoss,0.844716,0.806340,0.886927,0.957218



Best architechture CE+DiceLoss (F1: 0.9109) 


In [ ]:
# Save EVERY architecture model to a folder (Base_CE, FocalLoss, DiceLoss, CE+DiceLoss, CRF, CRF+FocalLoss)
# Uses your existing training logic, but adds export_model() and calls it for each arch.
#
# Assumes you already executed previous cells that define:
# dataset, unique_tags, id2label, label2id, compute_metrics,
# FocalLossTrainer, DiceLossTrainer, CEDiceLossTrainer, TransformerCRF

import os, json
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

# ====== Your exporter (kept close to what you posted) ======
def export_model(model, tokenizer, arch_type, save_dir, id2label, label2id):
    os.makedirs(save_dir, exist_ok=True)
    print(f"\n💾 Saving model ({arch_type}) to: {save_dir} ...")

    # 1) Save tokenizer (+ vocab if available)
    tokenizer.save_pretrained(save_dir)
    if hasattr(tokenizer, "save_vocabulary"):
        tokenizer.save_vocabulary(save_dir)

    # 2) Ensure label mapping stored in config.json
    if hasattr(model, "config") and model.config is not None:
        model.config.id2label = {int(k): v for k, v in id2label.items()} if isinstance(next(iter(id2label.keys())), int) else id2label
        model.config.label2id = label2id
        model.config.save_pretrained(save_dir)

    # 3) Save weights
    if arch_type in ["CRF", "CRF+FocalLoss"]:
        torch.save(model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))
        with open(os.path.join(save_dir, "labels.json"), "w", encoding="utf-8") as f:
            json.dump({"id2label": {str(k): v for k, v in id2label.items()}, "label2id": label2id}, f, ensure_ascii=False, indent=2)
        print("✅ Saved: Tokenizer + CRF Weights (pytorch_model.bin) + Config + labels.json")
    else:
        # If you want .bin instead of safetensors, set safe_serialization=False
        model.save_pretrained(save_dir, safe_serialization=True)
        print("✅ Saved: Tokenizer + HF Model + Config")

    print("📦 Files:", sorted(os.listdir(save_dir)))


# ====== Train + export for one arch ======
def train_and_export_one(model_name, arch_type, root_save_dir="./saved_models_all"):
    base_name = model_name.split("-")[0].upper()
    arch_name = f"{base_name}_{arch_type}".replace("+", "_").replace(" ", "_")
    save_dir = os.path.join(root_save_dir, arch_name)

    print(f"\n==============================")
    print(f"Training: {model_name} | Arch: {arch_type}")
    print(f"Will save to: {save_dir}")
    print(f"==============================")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        add_prefix_space=True if "roberta" in model_name else False
    )

    def tokenize_and_align_labels(examples):
        tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        labels = []
        for i, label in enumerate(examples["ner_tags_id"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            previous_word_idx = None
            label_ids = []
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(label[word_idx])
                else:
                    label_ids.append(-100)
                previous_word_idx = word_idx
            labels.append(label_ids)
        tokenized_inputs["labels"] = labels
        return tokenized_inputs

    tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    # pick model + trainer
    if arch_type in ["Base_CE", "FocalLoss", "DiceLoss", "CE+DiceLoss"]:
        model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(unique_tags),
            id2label=id2label,
            label2id=label2id
        )
        if arch_type == "Base_CE":
            trainer_class = Trainer
        elif arch_type == "FocalLoss":
            trainer_class = FocalLossTrainer
        elif arch_type == "DiceLoss":
            trainer_class = DiceLossTrainer
        else:  # "CE+DiceLoss"
            trainer_class = CEDiceLossTrainer

    elif arch_type == "CRF":
        model = TransformerCRF(model_name, num_labels=len(unique_tags), use_focal_loss=False)
        trainer_class = Trainer

    elif arch_type == "CRF+FocalLoss":
        model = TransformerCRF(model_name, num_labels=len(unique_tags), use_focal_loss=True, gamma=2.0)
        trainer_class = Trainer
    else:
        raise ValueError(f"Unknown arch_type: {arch_type}")

    training_args = TrainingArguments(
        output_dir=f"./temp_{arch_name}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
    )

    trainer = trainer_class(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_results = trainer.evaluate()
    print("Eval:", eval_results)

    # save metrics.json too (optional, nice to have)
    export_model(model=trainer.model, tokenizer=tokenizer, arch_type=arch_type,
                 save_dir=save_dir, id2label=id2label, label2id=label2id)

    with open(os.path.join(save_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(eval_results, f, ensure_ascii=False, indent=2)

    return eval_results


# ====== Run all archs and export all ======
ROOT_SAVE_DIR = "./saved_models_all"
VALID_ARCHS = ["Base_CE", "FocalLoss", "DiceLoss", "CE+DiceLoss", "CRF", "CRF+FocalLoss"]

all_metrics = {}
for arch in VALID_ARCHS:
    try:
        all_metrics[arch] = train_and_export_one(CHOSEN_MODEL, arch, root_save_dir=ROOT_SAVE_DIR)
    except Exception as e:
        print(f"❌ Error training/saving {arch}: {e}")

print("\n✅ Done. Saved folders:", sorted(os.listdir(ROOT_SAVE_DIR)) if os.path.exists(ROOT_SAVE_DIR) else "ROOT_SAVE_DIR not found")


Training: roberta-base | Arch: Base_CE
Will save to: ./saved_models_all/ROBERTA_Base_CE


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,5.002733,1.867575,0.892262,0.913481,0.902747,0.971592
2,1.652181,1.712966,0.908669,0.914502,0.911576,0.974213
3,1.298185,1.665888,0.906447,0.923111,0.914703,0.975396
4,1.160933,1.669364,0.915019,0.925299,0.920131,0.976347
5,0.915180,1.688146,0.919472,0.924570,0.922014,0.977235


Eval: {'eval_loss': 1.6881464719772339, 'eval_precision': 0.9194718514219384, 'eval_recall': 0.9245695943974321, 'eval_f1': 0.9220136767059508, 'eval_accuracy': 0.9772352568167406, 'eval_runtime': 9.0047, 'eval_samples_per_second': 244.428, 'eval_steps_per_second': 15.325, 'epoch': 5.0}

💾 Saving model (CRF) to: ./saved_models_all/ROBERTA_CRF ...
✅ Saved: Tokenizer + CRF Weights (pytorch_model.bin) + Config + labels.json
📦 Files: ['config.json', 'labels.json', 'pytorch_model.bin', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json']

Training: roberta-base | Arch: CRF+FocalLoss
Will save to: ./saved_models_all/ROBERTA_CRF_FocalLoss


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


In [14]:
!zip -r saved_models.zip saved_models_all

  adding: saved_models_all/ (stored 0%)
  adding: saved_models_all/DISTILBERT_DiceLoss/ (stored 0%)
  adding: saved_models_all/DISTILBERT_DiceLoss/config.json (deflated 55%)
  adding: saved_models_all/DISTILBERT_DiceLoss/metrics.json (deflated 42%)
  adding: saved_models_all/DISTILBERT_DiceLoss/tokenizer.json (deflated 70%)
  adding: saved_models_all/DISTILBERT_DiceLoss/tokenizer_config.json (deflated 43%)
  adding: saved_models_all/DISTILBERT_DiceLoss/tokenizer.model (deflated 49%)
  adding: saved_models_all/DISTILBERT_DiceLoss/model.safetensors (deflated 8%)
  adding: saved_models_all/DISTILBERT_FocalLoss/ (stored 0%)
  adding: saved_models_all/DISTILBERT_FocalLoss/config.json (deflated 55%)
  adding: saved_models_all/DISTILBERT_FocalLoss/metrics.json (deflated 42%)
  adding: saved_models_all/DISTILBERT_FocalLoss/tokenizer.json (deflated 70%)
  adding: saved_models_all/DISTILBERT_FocalLoss/tokenizer_config.json (deflated 43%)
  adding: saved_models_all/DISTILBERT_FocalLoss/tokenizer.